# 2. Modelagem Supervisionada — Previsão de Atrasos de Voos

## Objetivo
Aplicar algoritmos de **classificação** e **regressão** para prever atrasos de voos nos EUA (2015).

- **Classificação**: O voo vai atrasar ≥ 15 minutos? (Sim/Não)  
- **Regressão**: Se atrasar, quantos minutos de atraso?

### Algoritmos comparados
| Tipo | Algoritmos |
|------|-----------|
| Classificação | Random Forest, XGBoost, Logistic Regression |
| Regressão | Random Forest Regressor, XGBoost Regressor |

### Métricas de avaliação
- **Classificação**: Accuracy, Precision, Recall, F1-Score, AUC-ROC
- **Regressão**: MAE, RMSE, R²

## 2.1 Setup e Carregamento dos Dados

Importamos os módulos do projeto e carregamos os dados já limpos com as features engenheiradas (período do dia, estação, feriados, etc.).

In [1]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_clean_data
from src.feature_engineering import prepare_features
from src.supervised import (
    run_classification, run_regression, encode_categoricals,
    split_data, FEATURE_COLS
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# Carregar dados
flights, airlines, airports = load_clean_data()
flights = prepare_features(flights)
print(f"Dataset: {flights.shape[0]:,} voos × {flights.shape[1]} colunas")
print(f"\nDistribuição da variável alvo (IS_DELAYED):")
print(flights["IS_DELAYED"].value_counts(normalize=True).map("{:.2%}".format))

Dataset: 5,714,008 voos × 51 colunas

Distribuição da variável alvo (IS_DELAYED):
IS_DELAYED
0    81.39%
1    18.61%
Name: proportion, dtype: str


### Verificação das features

Antes de modelar, vamos confirmar que as features categóricas e numéricas estão presentes e válidas.

In [2]:
# Features que serão usadas nos modelos
cat_cols = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "PERIODO_DIA", "ESTACAO"]
num_cols = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE", "SCHEDULED_TIME",
            "IS_WEEKEND", "IS_HOLIDAY", "NEAR_HOLIDAY"]

print("Features categóricas:")
for c in cat_cols:
    print(f"  {c}: {flights[c].nunique()} valores únicos")

print(f"\nFeatures numéricas:")
for c in num_cols:
    print(f"  {c}: min={flights[c].min()}, max={flights[c].max()}, null={flights[c].isnull().sum()}")

print(f"\nTarget (IS_DELAYED): {flights['IS_DELAYED'].dtype}")
print(f"Target (ARRIVAL_DELAY): {flights['ARRIVAL_DELAY'].describe().round(2).to_dict()}")

Features categóricas:
  AIRLINE: 14 valores únicos


  ORIGIN_AIRPORT: 628 valores únicos
  DESTINATION_AIRPORT: 629 valores únicos


  PERIODO_DIA: 4 valores únicos
  ESTACAO: 4 valores únicos

Features numéricas:
  MONTH: min=1, max=12, null=0
  DAY_OF_WEEK: min=1, max=7, null=0
  DEP_HOUR: min=0, max=23, null=0
  DISTANCE: min=31, max=4983, null=0


  SCHEDULED_TIME: min=18.0, max=718.0, null=0
  IS_WEEKEND: min=0, max=1, null=0
  IS_HOLIDAY: min=0, max=1, null=0
  NEAR_HOLIDAY: min=0, max=1, null=0

Target (IS_DELAYED): int64


Target (ARRIVAL_DELAY): {'count': 5714008.0, 'mean': 4.41, 'std': 39.27, 'min': -87.0, '25%': -13.0, '50%': -5.0, '75%': 8.0, 'max': 1971.0}


## 2.2 Classificação — O voo vai atrasar?

**Objetivo**: Prever se um voo terá atraso ≥ 15 minutos na chegada (variável binária `IS_DELAYED`).

**Algoritmos**:
1. **Random Forest** (n_estimators=200, max_depth=15)
2. **XGBoost** (n_estimators=200, max_depth=8, lr=0.1)
3. **Logistic Regression** (max_iter=1000)

**Métricas**: Accuracy, Precision, Recall, F1-Score, AUC-ROC, Matriz de Confusão

> *Nota*: Para datasets > 500K registros, é feita amostragem para viabilizar o treinamento.

In [3]:
# Executar pipeline de classificação (3 modelos comparados)
clf_results = run_classification(flights)


Treinando: Random Forest


              precision    recall  f1-score   support

           0       0.87      0.70      0.78    930114
           1       0.29      0.54      0.38    212688

    accuracy                           0.67   1142802
   macro avg       0.58      0.62      0.58   1142802
weighted avg       0.76      0.67      0.70   1142802

AUC-ROC: 0.6717
CV AUC-ROC (3-fold): 0.6394 ± 0.0028

Treinando: XGBoost


              precision    recall  f1-score   support

           0       0.88      0.65      0.75    930114
           1       0.29      0.63      0.39    212688

    accuracy                           0.64   1142802
   macro avg       0.59      0.64      0.57   1142802
weighted avg       0.77      0.64      0.68   1142802

AUC-ROC: 0.6884
CV AUC-ROC (3-fold): 0.6410 ± 0.0017

Treinando: Logistic Regression


              precision    recall  f1-score   support

           0       0.87      0.53      0.66    930114
           1       0.24      0.64      0.35    212688

    accuracy                           0.55   1142802
   macro avg       0.55      0.59      0.50   1142802
weighted avg       0.75      0.55      0.60   1142802

AUC-ROC: 0.6168
CV AUC-ROC (3-fold): 0.6109 ± 0.0040

>> Melhor modelo (classificação): XGBoost (AUC=0.6884)


  >> Salvo: outputs/figures/classification_results.png


  >> Salvo: outputs/figures/feature_importance_clf.png


### Comparação dos Classificadores

Tabela resumo com as métricas de cada algoritmo. O melhor modelo é salvo em `outputs/models/best_classifier.pkl`.

In [4]:
# Tabela comparativa de classificação (inclui cross-validation)
clf_comparison = pd.DataFrame({
    name: {
        "Accuracy": r["report"]["accuracy"],
        "Precision (delayed)": r["report"]["1"]["precision"],
        "Recall (delayed)": r["report"]["1"]["recall"],
        "F1-Score (delayed)": r["report"]["1"]["f1-score"],
        "AUC-ROC": r["auc"],
        "CV AUC (3-fold)": r["cv_auc_mean"],
        "CV AUC std": r["cv_auc_std"],
    }
    for name, r in clf_results.items()
}).T.round(4)

best_clf = clf_comparison["AUC-ROC"].idxmax()
print(f"Melhor classificador (AUC-ROC): {best_clf}")
print(f"AUC-ROC = {clf_comparison.loc[best_clf, 'AUC-ROC']:.4f}")
print(f"CV AUC-ROC = {clf_comparison.loc[best_clf, 'CV AUC (3-fold)']:.4f} +/- {clf_comparison.loc[best_clf, 'CV AUC std']:.4f}\n")
clf_comparison.style.highlight_max(axis=0, color="lightgreen")

Melhor classificador (AUC-ROC): XGBoost
AUC-ROC = 0.6884
CV AUC-ROC = 0.6410 +/- 0.0017



,Accuracy,Precision (delayed),Recall (delayed),F1-Score (delayed),AUC-ROC,CV AUC (3-fold),CV AUC std
Random Forest,0.670700,0.291500,0.538000,0.378100,0.671700,0.639400,0.002800
XGBoost,0.642800,0.288300,0.625900,0.394700,0.688400,0.641000,0.001700
Logistic Regression,0.553300,0.239400,0.643300,0.349000,0.616800,0.610900,0.004000


## 2.3 Regressão — Quantos minutos de atraso?

**Objetivo**: Para voos que efetivamente atrasaram (ARRIVAL_DELAY > 0), prever quantos minutos de atraso.

**Algoritmos**:
1. **Random Forest Regressor** (n_estimators=150, max_depth=15)
2. **XGBoost Regressor** (n_estimators=200, max_depth=8, lr=0.1)

**Métricas**: MAE (erro médio absoluto), RMSE (raiz do erro quadrático médio), R² (coeficiente de determinação)

> *Nota*: Filtramos apenas voos com atraso > 0 min para a regressão ser mais realista.

In [5]:
# Executar pipeline de regressão (2 modelos comparados)
reg_results = run_regression(flights)


Treinando: Random Forest Regressor


MAE:  30.34 minutos
RMSE: 51.98 minutos
R²:   0.0189
CV MAE (3-fold): 31.00 ± 0.07

Treinando: XGBoost Regressor


MAE:  29.85 minutos
RMSE: 51.47 minutos
R²:   0.0381
CV MAE (3-fold): 30.79 ± 0.07

>> Melhor modelo (regressão): XGBoost Regressor (MAE=29.85)


  >> Salvo: outputs/figures/regression_results.png


### Comparação dos Regressores

Tabela resumo com as métricas de cada algoritmo. O melhor modelo é salvo em `outputs/models/best_regressor.pkl`.

In [6]:
# Tabela comparativa de regressão (inclui cross-validation)
reg_comparison = pd.DataFrame({
    name: {
        "MAE (min)": r["mae"],
        "RMSE (min)": r["rmse"],
        "R2": r["r2"],
        "CV MAE (3-fold)": r["cv_mae_mean"],
        "CV MAE std": r["cv_mae_std"],
    }
    for name, r in reg_results.items()
}).T.round(4)

best_reg = reg_comparison["MAE (min)"].idxmin()
print(f"Melhor regressor (MAE): {best_reg}")
print(f"MAE = {reg_comparison.loc[best_reg, 'MAE (min)']:.2f} minutos")
print(f"CV MAE = {reg_comparison.loc[best_reg, 'CV MAE (3-fold)']:.2f} +/- {reg_comparison.loc[best_reg, 'CV MAE std']:.2f}\n")
reg_comparison.style.highlight_min(subset=["MAE (min)", "RMSE (min)"], color="lightgreen")\
    .highlight_max(subset=["R2"], color="lightgreen")

Melhor regressor (MAE): XGBoost Regressor
MAE = 29.85 minutos
CV MAE = 30.79 +/- 0.07



,MAE (min),RMSE (min),R2,CV MAE (3-fold),CV MAE std
Random Forest Regressor,30.335100,51.979200,0.018900,30.996100,0.065900
XGBoost Regressor,29.847000,51.468300,0.038100,30.791400,0.071600


## 2.4 Analise Critica — Modelagem Supervisionada

### Interpretacao dos Resultados

**Classificacao**:
- Comparamos 3 algoritmos distintos (Random Forest, XGBoost, Logistic Regression)
- AUC-ROC como criterio principal — lida melhor com desbalanceamento de classes (~81/19)
- **Balanceamento de classes**: Aplicamos `class_weight='balanced'` (RF, LR) e `scale_pos_weight` (XGBoost) para lidar com o desbalanceamento (81% nao atrasados vs 19% atrasados). Isso elevou o **recall de voos atrasados de ~3% para ~63%** (XGBoost), permitindo detectar efetivamente os atrasos
- **Cross-validation 3-fold** aplicada para validar estabilidade: CV AUC proximo do AUC de teste indica boa generalizacao
- O trade-off: accuracy caiu de 82% para 64%, mas o modelo agora e muito mais util — detectar atrasos e mais importante do que acertar a maioria (que sao nao-atrasados)
- Feature Importance revela que DEP_HOUR, AIRLINE e MONTH sao as variaveis mais preditivas
- Matriz de Confusao mostra o trade-off entre falsos positivos e falsos negativos — com balanceamento, o modelo erra mais nos nao-atrasados mas acerta muito mais nos atrasados
- Features de congestionamento (FLIGHTS_SAME_HOUR_ORIGIN, ROUTE_POPULARITY) contribuem para a predicao

**Regressao**:
- Filtramos apenas voos com atraso > 0 para evitar vies
- MAE fornece interpretacao direta: "o modelo erra em media ~30 minutos"
- **Cross-validation 3-fold** confirma que o MAE de teste e robusto e nao sofre overfitting
- O R² baixo (0.038) indica que prever o tempo exato de atraso e extremamente dificil sem variaveis externas (clima, eventos). Porem o MAE de ~30 min da uma estimativa util da ordem de grandeza
- Grafico Real vs Previsto — pontos proximos a diagonal = bom modelo
- Distribuicao de residuos centrada em zero indica ausencia de vies sistematico
- Para atrasos extremos (>120 min), os regressores tendem a subestimar — esperado pelo efeito dos outliers

### Limitacoes
- Features como clima em tempo real e eventos especiais nao estao disponiveis — incorpora-los melhoraria significativamente os modelos
- Variaveis de detalhe de atraso (AIRLINE_DELAY, WEATHER_DELAY etc.) so existem para voos atrasados — nao podem ser usadas como preditoras (leakage)
- Amostragem (500K para clf, 300K para reg) pode perder padroes de subgrupos minoritarios
- Modelo treinado apenas com dados de 2015 — nao ha garantia de generalizacao temporal
- Tecnicas adicionais como SMOTE poderiam ser exploradas para complementar o balanceamento via pesos